# 02. Time-Series Demand Forecasting: Prophet vs. Baseline Regression
### AI-Based Clothing Sales Forecasting & Inventory Optimization System

This notebook provides a comparative machine learning study evaluating:
1. **Primary Model**: Facebook Prophet (Additive seasonality + promotional regressors)
2. **Baseline Model**: Scikit-Learn Ordinary Least Squares Linear Regression (Time index + cyclical calendar harmonics)

#### Evaluation Metrics:
- Root Mean Squared Error (RMSE)
- Mean Absolute Percentage Error (MAPE)
- Mean Absolute Error (MAE)
- Generalization check: Train vs. Test split metrics to diagnose overfitting.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from src.forecasting.prophet_model import forecast_demand
from src.forecasting.baseline_regression import forecast_baseline
from src.evaluation.metrics import calculate_forecasting_metrics

print("Forecasting modules imported successfully.")

## 1. Load Feature Matrix & Select Sample SKU (`OUT-001`)

In [ ]:
feat_path = '../data/processed/feature_matrix.csv' if os.path.exists('../data/processed/feature_matrix.csv') else 'data/processed/feature_matrix.csv'
df_features = pd.read_csv(feat_path)

sku = 'OUT-001'
print(f"Evaluating demand forecasting for {sku}...")

# Run Prophet
df_prophet_fc, prophet_rep = forecast_demand(sku, horizon_weeks=8, df_features=df_features)

# Run Baseline Linear Regression
df_base_fc, base_rep = forecast_baseline(sku, horizon_weeks=8, df_features=df_features)

print("Prophet Test Metrics:", prophet_rep['test_metrics'])
print("Baseline Test Metrics:", base_rep['test_metrics'])

## 2. Model Performance Benchmark Comparison Table

In [ ]:
metrics_comparison = pd.DataFrame([
    {
        'Model': 'Prophet (Primary)',
        'Train RMSE': prophet_rep['train_metrics']['rmse'],
        'Test RMSE': prophet_rep['test_metrics']['rmse'],
        'Train MAPE (%)': prophet_rep['train_metrics']['mape'],
        'Test MAPE (%)': prophet_rep['test_metrics']['mape'],
        'Test MAE': prophet_rep['test_metrics']['mae']
    },
    {
        'Model': 'Baseline Linear Regression',
        'Train RMSE': base_rep['train_metrics']['rmse'],
        'Test RMSE': base_rep['test_metrics']['rmse'],
        'Train MAPE (%)': base_rep['train_metrics']['mape'],
        'Test MAPE (%)': base_rep['test_metrics']['mape'],
        'Test MAE': base_rep['test_metrics']['mae']
    }
])
metrics_comparison

## 3. Forecast Comparison Plot with Confidence Bounds

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(pd.to_datetime(df_prophet_fc['date']), df_prophet_fc['predicted_quantity'], label='Prophet Forecast', color='#6366f1', lw=2.5)
plt.fill_between(pd.to_datetime(df_prophet_fc['date']), df_prophet_fc['lower_bound'], df_prophet_fc['upper_bound'], color='#6366f1', alpha=0.2, label='Prophet 95% Confidence Interval')
plt.plot(pd.to_datetime(df_base_fc['date']), df_base_fc['predicted_quantity'], label='Baseline Linear Regression', color='#f59e0b', linestyle='--', lw=2)

plt.title(f'8-Week Out-of-Sample Demand Forecast Comparison ({sku})', fontsize=14, fontweight='bold')
plt.xlabel('Forecast Date')
plt.ylabel('Daily Units Predicted')
plt.legend(frameon=True)
plt.tight_layout()
plt.show()